# Manager Evaluation v2

Comparison of strategies across two universes:

| Strategy | Description | Coverage |
|---|---|---|
| **Manager v1** | Fundamental analysis only | 5 stocks |
| **Manager v2 (legacy)** | Manager v2 previous version | 5 stocks |
| **Manager v2 (sem fato relevante)** | Fundamental analysis (simplified) | 9 stocks |
| **Manager v2 (GPT-5-mini)** | Fundamental analysis + material facts (GPT-5-mini) | 9 stocks |
| **Manager v2 (sabiazinho-4)** | Fundamental analysis + material facts (sabiazinho-4) | 9 stocks |
| **Manager v2 (hybrid)** | Hybrid facts (sabiazinho-4) + gesture (GPT-5-mini) | 9 stocks |
| **Baseline** | Price-only manager | 5 stocks |
| **Buy & Hold** | Benchmark: buy on day 1, hold until last day | 9 stocks |

⚠️ **Data Status:**
- Manager v1, v2 legacy & Baseline: Complete for 5 stocks
- Manager v2 variants & B&H: Complete for 9 stocks

In [11]:
import json
import numpy as np
from tabulate import tabulate

# Legacy paths (5 stocks)
RESULTS_V1 = "results/manager"
RESULTS_V2_LEGACY = "results/manager_v2"
RESULTS_BASE = "results/baseline"

# New paths with v2 models (9 stocks)
RESULTS_V2_SEM_FATO = "results_v2/sem_fato_relevante_gpt5mini"
RESULTS_V2_COMPLETO_GPT5 = "results_v2/completo_gpt5mini"
RESULTS_V2_COMPLETO_SABIAZINHO = "results_v2/completo_sabiazinho4"
RESULTS_V2_HIBRIDO = "results_v2/hibrido_fatos_sabiazinho4_gestor_gpt5mini"

## Helper Functions

In [12]:
def compute_portfolio_returns(decisions: list, results: list, stocks_filter: set = None) -> dict:
    """Simulate portfolio trades and return realized trade returns per stock.

    For each decision (sorted by analysis_date):
    - 'Comprar' opens (or adds to) a position at the day's closing price.
    - 'Vender' closes all open positions; trade return = (sell - avg_buy) / avg_buy.
    Remaining open positions are closed at the last observed price.

    Parameters
    ----------
    decisions : list
        List of decision dicts with keys: stock_id, analysis_date, recommendation
    results : list
        List of price records with keys: ACAO, DATA_DO_PREGAO, PRECO_ULTIMO_NEGOCIO
    stocks_filter : set, optional
        If provided, only include these stocks in the results

    Returns
    -------
    dict[str, list[float]]
        Realized trade returns per stock ticker.
    """
    # Build a lookup: (stock_id, date) -> closing price
    price_lookup = {(r["ACAO"], r["DATA_DO_PREGAO"]): r["PRECO_ULTIMO_NEGOCIO"] for r in results}

    open_positions: dict[str, list] = {}
    trade_returns: dict[str, list] = {}
    last_price: dict[str, float] = {}

    for decision in decisions:
        stock_id = decision["stock_id"]

        # Skip if not in filter
        if stocks_filter and stock_id not in stocks_filter:
            continue

        date = decision["analysis_date"]
        rec = decision["recommendation"]
        price = price_lookup[(stock_id, date)]

        last_price[stock_id] = price
        open_positions.setdefault(stock_id, [])
        trade_returns.setdefault(stock_id, [])

        if rec == "Comprar":
            open_positions[stock_id].append(price)
        elif rec == "Vender" and open_positions[stock_id]:
            avg_buy = np.mean(open_positions[stock_id])
            trade_returns[stock_id].append((price - avg_buy) / avg_buy)
            open_positions[stock_id] = []

    # Close any remaining open positions at last known price
    for stock_id, positions in open_positions.items():
        if positions:
            avg_buy = np.mean(positions)
            trade_returns[stock_id].append((last_price[stock_id] - avg_buy) / avg_buy)

    return trade_returns

In [13]:
def compute_buy_and_hold(results: list, stocks_filter: set = None) -> dict:
    """Compute buy-and-hold return per stock.

    Buys at the first observed closing price and sells at the last.

    Parameters
    ----------
    results : list
        List of price records
    stocks_filter : set, optional
        If provided, only include these stocks

    Returns
    -------
    dict[str, float]
        Total return per stock ticker.
    """
    prices_per_stock: dict[str, list] = {}
    for r in results:
        stock_id = r["ACAO"]

        if stocks_filter and stock_id not in stocks_filter:
            continue

        prices_per_stock.setdefault(stock_id, [])
        prices_per_stock[stock_id].append(r["PRECO_ULTIMO_NEGOCIO"])

    return {
        stock_id: (prices[-1] - prices[0]) / prices[0]
        for stock_id, prices in prices_per_stock.items()
        if len(prices) >= 2
    }


def total_return(trade_returns: dict) -> float:
    """Sum all realized trade returns across all stocks."""
    return sum(sum(v) for v in trade_returns.values())


def total_bnh_return(bnh: dict) -> float:
    """Sum buy-and-hold returns across all stocks."""
    return sum(bnh.values())

## Load Data

In [14]:
def load(folder: str) -> tuple:
    with open(f"{folder}/decisions_sample.json") as f:
        decisions = json.load(f)
    with open(f"{folder}/results_sample.json") as f:
        results = json.load(f)
    return decisions, results


# Load legacy data (5 stocks)
decisions_v1, results_v1 = load(RESULTS_V1)
decisions_v2_legacy, results_v2_legacy = load(RESULTS_V2_LEGACY)
decisions_base, results_base = load(RESULTS_BASE)

# Load v2 models (9 stocks)
decisions_v2_sem_fato, results_v2_sem_fato = load(RESULTS_V2_SEM_FATO)
decisions_v2_completo_gpt5, results_v2_completo_gpt5 = load(RESULTS_V2_COMPLETO_GPT5)
decisions_v2_completo_sabiazinho, results_v2_completo_sabiazinho = load(
    RESULTS_V2_COMPLETO_SABIAZINHO
)
decisions_v2_hibrido, results_v2_hibrido = load(RESULTS_V2_HIBRIDO)

print(
    f"Manager v1                      : {len(decisions_v1)} decisions, {len(results_v1)} price records"
)
print(
    f"Manager v2 (legacy)             : {len(decisions_v2_legacy)} decisions, {len(results_v2_legacy)} price records"
)
print(
    f"Baseline                        : {len(decisions_base)} decisions, {len(results_base)} price records"
)
print()
print(
    f"Manager v2 (sem fato)           : {len(decisions_v2_sem_fato)} decisions, {len(results_v2_sem_fato)} price records"
)
print(
    f"Manager v2 (completo GPT-5)     : {len(decisions_v2_completo_gpt5)} decisions, {len(results_v2_completo_gpt5)} price records"
)
print(
    f"Manager v2 (completo sabiazinho): {len(decisions_v2_completo_sabiazinho)} decisions, {len(results_v2_completo_sabiazinho)} price records"
)
print(
    f"Manager v2 (hibrido)            : {len(decisions_v2_hibrido)} decisions, {len(results_v2_hibrido)} price records"
)

Manager v1                      : 119 decisions, 120 price records
Manager v2 (legacy)             : 119 decisions, 120 price records
Baseline                        : 120 decisions, 120 price records

Manager v2 (sem fato)           : 215 decisions, 216 price records
Manager v2 (completo GPT-5)     : 214 decisions, 216 price records
Manager v2 (completo sabiazinho): 216 decisions, 216 price records
Manager v2 (hibrido)            : 216 decisions, 216 price records


## Identify Stock Universes

In [15]:
# Get all stocks in each strategy
stocks_v1 = set(d["stock_id"] for d in decisions_v1)
stocks_v2_legacy = set(d["stock_id"] for d in decisions_v2_legacy)
stocks_base = set(d["stock_id"] for d in decisions_base)

stocks_v2_sem_fato = set(d["stock_id"] for d in decisions_v2_sem_fato)
stocks_v2_completo_gpt5 = set(d["stock_id"] for d in decisions_v2_completo_gpt5)
stocks_v2_completo_sabiazinho = set(d["stock_id"] for d in decisions_v2_completo_sabiazinho)
stocks_v2_hibrido = set(d["stock_id"] for d in decisions_v2_hibrido)

# 5-stock universe: intersection of legacy strategies
common_stocks_5 = sorted(stocks_v1 & stocks_v2_legacy & stocks_base)

# 9-stock universe: union of all v2 models
common_stocks_9 = sorted(
    stocks_v2_sem_fato | stocks_v2_completo_gpt5 | stocks_v2_completo_sabiazinho | stocks_v2_hibrido
)

print(f"5-stock universe (legacy):      {common_stocks_5}")
print(f"9-stock universe (v2 models):   {common_stocks_9}")
print(f"Overlap:                        {sorted(set(common_stocks_5) & set(common_stocks_9))}")

5-stock universe (legacy):      ['ALUP11', 'AURE3', 'CPLE3', 'ENEV3', 'EQTL3']
9-stock universe (v2 models):   ['ALUP11', 'AURE3', 'CPLE3', 'EALT4', 'ENEV3', 'EQTL3', 'PETR4', 'RECV3', 'VALE3']
Overlap:                        ['ALUP11', 'AURE3', 'CPLE3', 'ENEV3', 'EQTL3']


---
# SECTION 1: 5-Stock Universe (Legacy Strategies)

In [16]:
# Compute returns for 5 common stocks
common_stocks_5_set = set(common_stocks_5)

returns_v1_5 = compute_portfolio_returns(decisions_v1, results_v1, common_stocks_5_set)
returns_v2_legacy_5 = compute_portfolio_returns(
    decisions_v2_legacy, results_v2_legacy, common_stocks_5_set
)
returns_v2_sem_fato_5 = compute_portfolio_returns(
    decisions_v2_sem_fato, results_v2_sem_fato, common_stocks_5_set
)
returns_v2_completo_gpt5_5 = compute_portfolio_returns(
    decisions_v2_completo_gpt5, results_v2_completo_gpt5, common_stocks_5_set
)
returns_v2_completo_sabiazinho_5 = compute_portfolio_returns(
    decisions_v2_completo_sabiazinho, results_v2_completo_sabiazinho, common_stocks_5_set
)
returns_v2_hibrido_5 = compute_portfolio_returns(
    decisions_v2_hibrido, results_v2_hibrido, common_stocks_5_set
)
returns_base_5 = compute_portfolio_returns(decisions_base, results_base, common_stocks_5_set)
bnh_5 = compute_buy_and_hold(results_v2_sem_fato, common_stocks_5_set)

print("✓ Returns computed for 5-stock universe (including v2 models)")

✓ Returns computed for 5-stock universe (including v2 models)


In [17]:
print("=" * 160)
print("PER-STOCK RETURNS (5-Stock Universe)")
print("=" * 160)

headers = [
    "Stock",
    "Manager v1",
    "Manager v2 (legacy)",
    "Manager v2 (sem fato)",
    "Manager v2 (GPT-5-mini)",
    "Manager v2 (sabiazinho-4)",
    "Manager v2 (hybrid)",
    "Baseline",
    "Buy & Hold",
]

rows = []
for stock in common_stocks_5:
    r1 = sum(returns_v1_5.get(stock, [0.0]))
    r2_legacy = sum(returns_v2_legacy_5.get(stock, [0.0]))
    r2_sem_fato = sum(returns_v2_sem_fato_5.get(stock, [0.0]))
    r2_gpt5 = sum(returns_v2_completo_gpt5_5.get(stock, [0.0]))
    r2_sabiazinho = sum(returns_v2_completo_sabiazinho_5.get(stock, [0.0]))
    r2_hibrido = sum(returns_v2_hibrido_5.get(stock, [0.0]))
    rb = sum(returns_base_5.get(stock, [0.0]))
    rbh = bnh_5.get(stock, 0.0)

    rows.append(
        [
            stock,
            f"{r1 * 100:.2f}%",
            f"{r2_legacy * 100:.2f}%",
            f"{r2_sem_fato * 100:.2f}%",
            f"{r2_gpt5 * 100:.2f}%",
            f"{r2_sabiazinho * 100:.2f}%",
            f"{r2_hibrido * 100:.2f}%",
            f"{rb * 100:.2f}%",
            f"{rbh * 100:.2f}%",
        ]
    )

# TOTAL row
rows.append(
    [
        "**TOTAL**",
        f"{total_return(returns_v1_5) * 100:.2f}%",
        f"{total_return(returns_v2_legacy_5) * 100:.2f}%",
        f"{total_return(returns_v2_sem_fato_5) * 100:.2f}%",
        f"{total_return(returns_v2_completo_gpt5_5) * 100:.2f}%",
        f"{total_return(returns_v2_completo_sabiazinho_5) * 100:.2f}%",
        f"{total_return(returns_v2_hibrido_5) * 100:.2f}%",
        f"{total_return(returns_base_5) * 100:.2f}%",
        f"{total_bnh_return(bnh_5) * 100:.2f}%",
    ]
)

print(tabulate(rows, headers=headers, tablefmt="github"))
print()

PER-STOCK RETURNS (5-Stock Universe)
| Stock     | Manager v1   | Manager v2 (legacy)   | Manager v2 (sem fato)   | Manager v2 (GPT-5-mini)   | Manager v2 (sabiazinho-4)   | Manager v2 (hybrid)   | Baseline   | Buy & Hold   |
|-----------|--------------|-----------------------|-------------------------|---------------------------|-----------------------------|-----------------------|------------|--------------|
| ALUP11    | 4.63%        | 16.12%                | 13.82%                  | 14.09%                    | 16.44%                      | 13.82%                | 17.99%     | 12.25%       |
| AURE3     | 18.75%       | 33.44%                | 16.07%                  | 56.67%                    | 1.39%                       | 15.57%                | 24.34%     | -8.12%       |
| CPLE3     | 25.34%       | 42.88%                | 46.87%                  | 43.09%                    | 20.48%                      | 42.88%                | 0.00%      | 46.87%       |
| ENEV3     | 115.

In [18]:
print("=" * 110)
print("AGGREGATE SUMMARY (5-Stock Universe)")
print("=" * 110)

summary_headers = ["Strategy", "Total Return", "Avg Return/Stock"]
summary_rows = [
    [
        "Manager v1 (fundamental only)",
        f"{total_return(returns_v1_5) * 100:.2f}%",
        f"{total_return(returns_v1_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (legacy)",
        f"{total_return(returns_v2_legacy_5) * 100:.2f}%",
        f"{total_return(returns_v2_legacy_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (sem fato relevante)",
        f"{total_return(returns_v2_sem_fato_5) * 100:.2f}%",
        f"{total_return(returns_v2_sem_fato_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (GPT-5-mini)",
        f"{total_return(returns_v2_completo_gpt5_5) * 100:.2f}%",
        f"{total_return(returns_v2_completo_gpt5_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (sabiazinho-4)",
        f"{total_return(returns_v2_completo_sabiazinho_5) * 100:.2f}%",
        f"{total_return(returns_v2_completo_sabiazinho_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (hybrid)",
        f"{total_return(returns_v2_hibrido_5) * 100:.2f}%",
        f"{total_return(returns_v2_hibrido_5) / 5 * 100:.2f}%",
    ],
    [
        "Baseline (price only)",
        f"{total_return(returns_base_5) * 100:.2f}%",
        f"{total_return(returns_base_5) / 5 * 100:.2f}%",
    ],
    [
        "Buy & Hold",
        f"{total_bnh_return(bnh_5) * 100:.2f}%",
        f"{total_bnh_return(bnh_5) / 5 * 100:.2f}%",
    ],
]

print(tabulate(summary_rows, headers=summary_headers, tablefmt="github"))

AGGREGATE SUMMARY (5-Stock Universe)
| Strategy                        | Total Return   | Avg Return/Stock   |
|---------------------------------|----------------|--------------------|
| Manager v1 (fundamental only)   | 183.98%        | 36.80%             |
| Manager v2 (legacy)             | 197.84%        | 39.57%             |
| Manager v2 (sem fato relevante) | 95.25%         | 19.05%             |
| Manager v2 (GPT-5-mini)         | 198.59%        | 39.72%             |
| Manager v2 (sabiazinho-4)       | 53.56%         | 10.71%             |
| Manager v2 (hybrid)             | 130.45%        | 26.09%             |
| Baseline (price only)           | 75.71%         | 15.14%             |
| Buy & Hold                      | 117.91%        | 23.58%             |


---
# SECTION 2: 9-Stock Universe (Results v2 Models)

In [19]:
# Compute returns for 9-stock universe
common_stocks_9_set = set(common_stocks_9)

returns_v2_sem_fato_9 = compute_portfolio_returns(
    decisions_v2_sem_fato, results_v2_sem_fato, common_stocks_9_set
)
returns_v2_completo_gpt5_9 = compute_portfolio_returns(
    decisions_v2_completo_gpt5, results_v2_completo_gpt5, common_stocks_9_set
)
returns_v2_completo_sabiazinho_9 = compute_portfolio_returns(
    decisions_v2_completo_sabiazinho, results_v2_completo_sabiazinho, common_stocks_9_set
)
returns_v2_hibrido_9 = compute_portfolio_returns(
    decisions_v2_hibrido, results_v2_hibrido, common_stocks_9_set
)
bnh_9 = compute_buy_and_hold(results_v2_sem_fato, common_stocks_9_set)

print("✓ Returns computed for 9-stock universe")

✓ Returns computed for 9-stock universe


In [20]:
print("=" * 140)
print("PER-STOCK RETURNS (9-Stock Universe)")
print("=" * 140)

headers_9 = [
    "Stock",
    "Manager v2 (sem fato)",
    "Manager v2 (GPT-5-mini)",
    "Manager v2 (sabiazinho-4)",
    "Manager v2 (hybrid)",
    "Buy & Hold",
]

rows_9 = []
for stock in common_stocks_9:
    r_sem_fato = sum(returns_v2_sem_fato_9.get(stock, [0.0]))
    r_gpt5 = sum(returns_v2_completo_gpt5_9.get(stock, [0.0]))
    r_sabiazinho = sum(returns_v2_completo_sabiazinho_9.get(stock, [0.0]))
    r_hibrido = sum(returns_v2_hibrido_9.get(stock, [0.0]))
    rbh = bnh_9.get(stock, 0.0)

    rows_9.append(
        [
            stock,
            f"{r_sem_fato * 100:.2f}%",
            f"{r_gpt5 * 100:.2f}%",
            f"{r_sabiazinho * 100:.2f}%",
            f"{r_hibrido * 100:.2f}%",
            f"{rbh * 100:.2f}%",
        ]
    )

# TOTAL row
rows_9.append(
    [
        "**TOTAL**",
        f"{total_return(returns_v2_sem_fato_9) * 100:.2f}%",
        f"{total_return(returns_v2_completo_gpt5_9) * 100:.2f}%",
        f"{total_return(returns_v2_completo_sabiazinho_9) * 100:.2f}%",
        f"{total_return(returns_v2_hibrido_9) * 100:.2f}%",
        f"{total_bnh_return(bnh_9) * 100:.2f}%",
    ]
)

print(tabulate(rows_9, headers=headers_9, tablefmt="github"))
print()

PER-STOCK RETURNS (9-Stock Universe)
| Stock     | Manager v2 (sem fato)   | Manager v2 (GPT-5-mini)   | Manager v2 (sabiazinho-4)   | Manager v2 (hybrid)   | Buy & Hold   |
|-----------|-------------------------|---------------------------|-----------------------------|-----------------------|--------------|
| ALUP11    | 13.82%                  | 14.09%                    | 16.44%                      | 13.82%                | 12.25%       |
| AURE3     | 16.07%                  | 56.67%                    | 1.39%                       | 15.57%                | -8.12%       |
| CPLE3     | 46.87%                  | 43.09%                    | 20.48%                      | 42.88%                | 46.87%       |
| EALT4     | 4.45%                   | 37.02%                    | 49.75%                      | 3.65%                 | 36.35%       |
| ENEV3     | 0.00%                   | 52.67%                    | -11.32%                     | 39.79%                | 54.63%       |
| EQ

In [21]:
print("=" * 110)
print("AGGREGATE SUMMARY (9-Stock Universe)")
print("=" * 110)

num_stocks_9 = len(common_stocks_9)
summary_headers_9 = ["Strategy", "Total Return", "Avg Return/Stock"]
summary_rows_9 = [
    [
        "Manager v2 (sem fato relevante)",
        f"{total_return(returns_v2_sem_fato_9) * 100:.2f}%",
        f"{total_return(returns_v2_sem_fato_9) / num_stocks_9 * 100:.2f}%",
    ],
    [
        "Manager v2 (GPT-5-mini)",
        f"{total_return(returns_v2_completo_gpt5_9) * 100:.2f}%",
        f"{total_return(returns_v2_completo_gpt5_9) / num_stocks_9 * 100:.2f}%",
    ],
    [
        "Manager v2 (sabiazinho-4)",
        f"{total_return(returns_v2_completo_sabiazinho_9) * 100:.2f}%",
        f"{total_return(returns_v2_completo_sabiazinho_9) / num_stocks_9 * 100:.2f}%",
    ],
    [
        "Manager v2 (hybrid)",
        f"{total_return(returns_v2_hibrido_9) * 100:.2f}%",
        f"{total_return(returns_v2_hibrido_9) / num_stocks_9 * 100:.2f}%",
    ],
    [
        "Buy & Hold",
        f"{total_bnh_return(bnh_9) * 100:.2f}%",
        f"{total_bnh_return(bnh_9) / num_stocks_9 * 100:.2f}%",
    ],
]

print(tabulate(summary_rows_9, headers=summary_headers_9, tablefmt="github"))

AGGREGATE SUMMARY (9-Stock Universe)
| Strategy                        | Total Return   | Avg Return/Stock   |
|---------------------------------|----------------|--------------------|
| Manager v2 (sem fato relevante) | 50.61%         | 5.62%              |
| Manager v2 (GPT-5-mini)         | 186.60%        | 20.73%             |
| Manager v2 (sabiazinho-4)       | 58.79%         | 6.53%              |
| Manager v2 (hybrid)             | 121.54%        | 13.50%             |
| Buy & Hold                      | 77.27%         | 8.59%              |


---
# SECTION 3: Cross-Universe Comparison

In [22]:
print("=" * 160)
print("CROSS-UNIVERSE PERFORMANCE SUMMARY")
print("=" * 160)

cross_headers = ["Strategy", "Universe", "Total Return", "# Stocks", "Avg Return/Stock"]
cross_rows = [
    [
        "Manager v1",
        "5-stock (legacy)",
        f"{total_return(returns_v1_5) * 100:.2f}%",
        "5",
        f"{total_return(returns_v1_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (legacy)",
        "5-stock (legacy)",
        f"{total_return(returns_v2_legacy_5) * 100:.2f}%",
        "5",
        f"{total_return(returns_v2_legacy_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (sem fato)",
        "5-stock (legacy)",
        f"{total_return(returns_v2_sem_fato_5) * 100:.2f}%",
        "5",
        f"{total_return(returns_v2_sem_fato_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (sem fato)",
        "9-stock (v2)",
        f"{total_return(returns_v2_sem_fato_9) * 100:.2f}%",
        "9",
        f"{total_return(returns_v2_sem_fato_9) / 9 * 100:.2f}%",
    ],
    [
        "Manager v2 (GPT-5-mini)",
        "5-stock (legacy)",
        f"{total_return(returns_v2_completo_gpt5_5) * 100:.2f}%",
        "5",
        f"{total_return(returns_v2_completo_gpt5_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (GPT-5-mini)",
        "9-stock (v2)",
        f"{total_return(returns_v2_completo_gpt5_9) * 100:.2f}%",
        "9",
        f"{total_return(returns_v2_completo_gpt5_9) / 9 * 100:.2f}%",
    ],
    [
        "Manager v2 (sabiazinho-4)",
        "5-stock (legacy)",
        f"{total_return(returns_v2_completo_sabiazinho_5) * 100:.2f}%",
        "5",
        f"{total_return(returns_v2_completo_sabiazinho_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (sabiazinho-4)",
        "9-stock (v2)",
        f"{total_return(returns_v2_completo_sabiazinho_9) * 100:.2f}%",
        "9",
        f"{total_return(returns_v2_completo_sabiazinho_9) / 9 * 100:.2f}%",
    ],
    [
        "Manager v2 (hybrid)",
        "5-stock (legacy)",
        f"{total_return(returns_v2_hibrido_5) * 100:.2f}%",
        "5",
        f"{total_return(returns_v2_hibrido_5) / 5 * 100:.2f}%",
    ],
    [
        "Manager v2 (hybrid)",
        "9-stock (v2)",
        f"{total_return(returns_v2_hibrido_9) * 100:.2f}%",
        "9",
        f"{total_return(returns_v2_hibrido_9) / 9 * 100:.2f}%",
    ],
    [
        "Baseline",
        "5-stock (legacy)",
        f"{total_return(returns_base_5) * 100:.2f}%",
        "5",
        f"{total_return(returns_base_5) / 5 * 100:.2f}%",
    ],
    [
        "Buy & Hold",
        "5-stock (legacy)",
        f"{total_bnh_return(bnh_5) * 100:.2f}%",
        "5",
        f"{total_bnh_return(bnh_5) / 5 * 100:.2f}%",
    ],
    [
        "Buy & Hold",
        "9-stock (v2)",
        f"{total_bnh_return(bnh_9) * 100:.2f}%",
        "9",
        f"{total_bnh_return(bnh_9) / 9 * 100:.2f}%",
    ],
]

print(tabulate(cross_rows, headers=cross_headers, tablefmt="github"))

CROSS-UNIVERSE PERFORMANCE SUMMARY
| Strategy                  | Universe         | Total Return   |   # Stocks | Avg Return/Stock   |
|---------------------------|------------------|----------------|------------|--------------------|
| Manager v1                | 5-stock (legacy) | 183.98%        |          5 | 36.80%             |
| Manager v2 (legacy)       | 5-stock (legacy) | 197.84%        |          5 | 39.57%             |
| Manager v2 (sem fato)     | 5-stock (legacy) | 95.25%         |          5 | 19.05%             |
| Manager v2 (sem fato)     | 9-stock (v2)     | 50.61%         |          9 | 5.62%              |
| Manager v2 (GPT-5-mini)   | 5-stock (legacy) | 198.59%        |          5 | 39.72%             |
| Manager v2 (GPT-5-mini)   | 9-stock (v2)     | 186.60%        |          9 | 20.73%             |
| Manager v2 (sabiazinho-4) | 5-stock (legacy) | 53.56%         |          5 | 10.71%             |
| Manager v2 (sabiazinho-4) | 9-stock (v2)     | 58.79%         |